# L6 — Infrastructure Design Translation

Synthesises L1–L4 stats into hardware and software sizing parameters (design_spec.md).

**Memory strategy**: trades loaded once; OB loaded per-exchange with 2 columns only.

In [ ]:
import os, gc, json, numpy as np, pandas as pd
from datetime import datetime, timedelta, timezone
import re

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_trades, load_orderbook, load_orderbook_updates
from mda.layer1.rates import layer1_summary
from mda.layer2.volleys import detect_volleys, volley_size_distribution
from mda.layer3.execution_rate import compute_execution_rate, execution_rate_stats
from mda.layer3.microbursts import detect_microbursts, microburst_stats
from mda.layer3.sequencing import compute_out_of_order
from mda.layer3.quantisation import resolution_report
from mda.layer4.update_rates import compute_update_rate_by_depth
from mda.layer6.spec import compute_specs, write_design_spec
from mda.plots.charts import save_figure
print("imports OK")

In [ ]:
# 2-hour window (same logic as L4)
ob_files = sorted(f for f in os.listdir(os.path.join(DATA_DIR, "orderbook")) if f.endswith(".parquet"))
first_ts = re.search(r'(\d{8}_\d{6})', ob_files[0]).group(1)
t0 = datetime.strptime(first_ts, "%Y%m%d_%H%M%S").replace(tzinfo=timezone.utc)
t2 = t0 + timedelta(hours=2)
START_DT = t0.strftime("%Y-%m-%dT%H:%M:%SZ")
END_DT   = t2.strftime("%Y-%m-%dT%H:%M:%SZ")
print(f"Window: {START_DT} → {END_DT}")

In [ ]:
df = load_trades(DATA_DIR)
EXCHANGES = df["exchange"].unique().tolist()
print(f"Loaded {len(df):,} trades | {EXCHANGES}")

In [ ]:
l1 = layer1_summary(df)
print("L1 done:", list(l1.keys()))

In [ ]:
_, volley_stats = detect_volleys(df)
l2 = {"volley_stats": volley_stats}
print(f"L2 done: {len(volley_stats):,} volleys")

In [ ]:
exec_rate  = compute_execution_rate(df)
exec_stats = execution_rate_stats(exec_rate)
bursts     = detect_microbursts(df)
mb_s       = microburst_stats(bursts)
oo_summary, _ = compute_out_of_order(df)
res_report = resolution_report(df)
l3 = {
    "exec_rate_stats": exec_stats,
    "microburst_stats": mb_s,
    "oo_summary": oo_summary,
    "resolution_report": res_report,
}
print("L3 done:", {k: len(v) for k,v in l3.items()})
del exec_rate, bursts
gc.collect()

In [ ]:
# OB update rate: 3-column load per exchange, 2-hour window
ob_rate_parts = []
for exch in EXCHANGES:
    print(f"  OB {exch}...")
    ob_exch = load_orderbook(DATA_DIR, exchanges=[exch],
                             columns=["exchange", "level", "received_at"],
                             add_ts_cols=False,
                             start_dt=START_DT, end_dt=END_DT)
    ob_exch["receive_ts_dt"] = pd.to_datetime(ob_exch["received_at"], utc=True, format="mixed")
    ob_rate_parts.append(compute_update_rate_by_depth(ob_exch))
    del ob_exch; gc.collect()

ob_update_rate = pd.concat(ob_rate_parts, ignore_index=True)
del ob_rate_parts; gc.collect()
l4 = {"ob_update_rate": ob_update_rate}
print("L4 done:", len(ob_update_rate), "rows")

In [ ]:
specs = compute_specs(l1, l2, l3, l4)
print(json.dumps(specs, indent=2))

In [ ]:
spec_path = os.path.join(REPORTS_DIR, "design_spec.md")
write_design_spec(specs, spec_path)
print(f"Written: {spec_path}")

In [ ]:
from IPython.display import Markdown, display
with open(spec_path) as f:
    display(Markdown(f.read()))

In [ ]:
pd.DataFrame(list(specs.items()), columns=["Parameter", "Value"])